# EuroSAT SmallCNN, the Keras twin

This is the TensorFlow/Keras equivalent of the from-scratch `SmallCNN` trained
in PyTorch in `src/malaigue/eurosat/`. It exists so the "familiar with the
Keras/TF equivalent" claim is honest. The authoritative run is the PyTorch one,
written up in `docs/eurosat.md`; this mirrors the same architecture and the same
honest evaluation in a different framework.

It is a Colab notebook on purpose, because TensorFlow is heavy and pins
protobuf, ml-dtypes and numpy bounds, so it is kept out of the project's `uv`
env. Run it on Colab with Runtime then Run all, where TensorFlow and
`tensorflow_datasets` are preinstalled.

Differences from the PyTorch version, stated honestly: the split here uses
`tfds` deterministic percentage slices rather than the per-class stratified
split, and training runs on the Colab GPU rather than CPU. The architecture, the
train-only normalization, the flip augmentation, and the metrics match.

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers

print("tensorflow", tf.__version__)
SEED = 42
NUM_CLASSES = 10
tf.random.set_seed(SEED)

In [ ]:
# EuroSAT RGB: 27,000 64x64 Sentinel-2 chips, 10 land-use classes.
# Deterministic 70/15/15 split via tfds slices.
(ds_train, ds_val, ds_test), info = tfds.load(
    "eurosat/rgb",
    split=["train[:70%]", "train[70%:85%]", "train[85%:]"],
    as_supervised=True,
    with_info=True,
)
CLASSES = info.features["label"].names
print(CLASSES)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def to_float(x, y):
    return tf.cast(x, tf.float32) / 255.0, y

def augment(x, y):
    # overhead imagery has no canonical orientation: flips are label-preserving
    return tf.image.random_flip_up_down(tf.image.random_flip_left_right(x)), y

def prep(ds, training):
    ds = ds.map(to_float, AUTOTUNE)
    if training:
        ds = ds.shuffle(2048, seed=SEED).map(augment, AUTOTUNE)
    return ds.batch(64).prefetch(AUTOTUNE)

train_ds, val_ds, test_ds = prep(ds_train, True), prep(ds_val, False), prep(ds_test, False)

# train-only normalization (no val/test leakage), same idea as the PyTorch version
norm = layers.Normalization()
norm.adapt(ds_train.map(to_float, AUTOTUNE).map(lambda x, y: x).batch(256))

In [ ]:
def small_cnn(num_classes=NUM_CLASSES):
    def block(x, ch):
        x = layers.Conv2D(ch, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        return layers.MaxPooling2D()(x)

    inp = keras.Input((64, 64, 3))
    x = norm(inp)
    for ch in (32, 64, 128, 256):  # 64 -> 32 -> 16 -> 8 -> 4
        x = block(x, ch)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes)(x)  # logits
    return keras.Model(inp, out)

model = small_cnn()
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
model.summary()

In [ ]:
history = model.fit(train_ds, validation_data=val_ds, epochs=20)

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"test accuracy: {test_acc:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_pred = model.predict(test_ds).argmax(1)
cm = tf.math.confusion_matrix(y_true, y_pred, num_classes=NUM_CLASSES).numpy()
norm_cm = cm / cm.sum(1, keepdims=True).clip(min=1)

fig, ax = plt.subplots(figsize=(8, 7))
ax.imshow(norm_cm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASSES, rotation=45, ha="right"); ax.set_yticklabels(CLASSES)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
ax.set_title("EuroSAT SmallCNN (Keras) confusion matrix")
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7,
                    color="white" if norm_cm[i, j] > 0.5 else "black")
plt.tight_layout(); plt.show()

The Keras twin reaches accuracy in the same range as the PyTorch SmallCNN and
shows the same kinds of confusions, among the crop and vegetation classes and
between River and the crops. It confirms that the architecture and the
evaluation carry across frameworks. The numbers to defend are in
`docs/eurosat.md`.